In [6]:
import pandas
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import DictVectorizer
from scipy.sparse import hstack
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

from keras.preprocessing import sequence
from keras.models import Sequential
from keras.preprocessing import text
from keras.utils import pad_sequences
import wandb

from keras import Sequential
import keras
from sklearn.ensemble import RandomForestRegressor

import tensorflow as tf



In [7]:
train = pandas.read_csv('../dataset/cleaner_dataset.csv')

train['tweet'] = train['tweet'].str.lower()
train['tweet'] = train['tweet'].replace('[^a-zA-Z0-9]', ' ', regex = True)

X = train['tweet']
y_n = train['neuroticism']
y_e = train['extraversion']
y_o = train['openness']
y_a = train['agreeableness']
y_c = train['conscientiousness']

In [8]:
wandb.init()
config = wandb.config

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: sahibzadahafeez (patsm). Use `wandb login --relogin` to force relogin


In [9]:


#set papameters
config.vocab_size = 1000
config.maxlen = 30
config.batch_size = 32
config.embedding_dims = 10
config.filters = 16
config.kernel_size = 3
config.hidden_dims = 250
config.epochs = 10

#X_train, X_test, y_train_n, y_test_n = train_test_split(X, y_n, test_size=0.2)

X_train, X_test, y_train_n, y_test_n, y_train_e, y_test_e, y_train_o, y_test_o, y_train_a, y_test_a, y_train_c, y_test_c = train_test_split(X, y_n, y_e, y_o, y_a, y_c, test_size=0.2)

tokenizer = text.Tokenizer(num_words = config.vocab_size)
tokenizer.fit_on_texts(X)
X_train = tokenizer.texts_to_matrix(X_train)
X_test = tokenizer.texts_to_matrix(X_test)

#X_train = pad_sequences(X_train, maxlen = config.maxlen)
#X_test = pad_sequences(X_test, maxlen = config.maxlen)

In [ ]:
#X_train, X_test, y_train_n, y_test_n, y_train_e, y_test_e, y_train_o, y_test_o, y_train_a, y_test_a, y_train_c, y_test_c = train_test_split(X, y_n, y_e, y_o, y_a, y_c, test_size=0.2)

In [10]:
sample_size = X_train.shape[0]
time_steps = X_train.shape[1]
input_dimention = 1

train_data_reshaped = X_train.reshape(sample_size, time_steps, input_dimention)

sample_size = X_test.shape[0]
time_steps = X_test.shape[1]
input_dimention = 1

test_data_reshaped = X_test.reshape(sample_size, time_steps, input_dimention)

In [11]:
n_timesteps = train_data_reshaped.shape[1]
n_features = train_data_reshaped.shape[2]

model = Sequential(name="model_conv1D")
model.add(keras.layers.Input(shape=(n_timesteps, n_features)))
model.add(keras.layers.Conv1D(filters=64, kernel_size=500, activation="relu", name="Conv1D_1"))
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.Conv1D(filters=32, kernel_size=100, activation="relu", name="Conv1D_2"))
model.add(keras.layers.Conv1D(filters=16, kernel_size=10, activation="relu", name="Conv1D_3"))
model.add(keras.layers.Conv1D(filters=8, kernel_size=2, activation="relu", name="Conv1D_4"))

model.add(keras.layers.MaxPooling1D(pool_size=2, name="MaxPooling1D"))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(32, activation="relu", name="Dense_1"))
model.add(keras.layers.Dense(n_features, name="Dense_2"))

optimizer = tf.keras.optimizers.RMSprop(0.001)

model.compile(loss='mse', optimizer=optimizer, metrics=['mae'])

model.summary()


Model: "model_conv1D"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 Conv1D_1 (Conv1D)           (None, 501, 64)           32064     
                                                                 
 dropout (Dropout)           (None, 501, 64)           0         
                                                                 
 Conv1D_2 (Conv1D)           (None, 402, 32)           204832    
                                                                 
 Conv1D_3 (Conv1D)           (None, 393, 16)           5136      
                                                                 
 Conv1D_4 (Conv1D)           (None, 392, 8)            264       
                                                                 
 MaxPooling1D (MaxPooling1D)  (None, 196, 8)           0         
                                                                 
 flatten (Flatten)           (None, 1568)             

In [ ]:
model.fit(train_data_reshaped, y_train_n, epochs=10, validation_split=0.2, verbose=1)

In [ ]:
model.fit(train_data_reshaped, y_train_e, epochs=10, validation_split=0.2, verbose=1)

In [ ]:
model.fit(train_data_reshaped, y_train_o, epochs=10, validation_split=0.2, verbose=1)

In [ ]:
model.fit(train_data_reshaped, y_train_a, epochs=10, validation_split=0.2, verbose=1)

In [ ]:
model.fit(train_data_reshaped, y_train_c, epochs=10, validation_split=0.2, verbose=1)